In [11]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.pipeline import Pipeline
from helpers import make_preprocessor



In [6]:
df = pd.read_csv("data/ames_housing.csv")
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
train_df = train_df.drop(index=[523, 1298])
X_train = train_df.drop(columns =['Id', 'SalePrice', 'GarageYrBlt', 'PoolArea'])
y_train = train_df['SalePrice']

In [10]:
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

models = {
    "XGBoost": XGBRegressor(random_state=42, n_jobs=-1),
    "LightGBM": LGBMRegressor(random_state=42, n_jobs=-1, verbose=-1),
    "CatBoost": CatBoostRegressor(random_state=42, verbose=0)
}

rows = []

for name, model in models.items():
    pipe = Pipeline([('prep', make_preprocessor(X_train)), ('model', model)])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring="neg_mean_absolute_error")
    rows.append({"Model": name, "MAE": -scores.mean(), "Spread": scores.std()})


pd.DataFrame(rows).sort_values("MAE").round(1)

,Model,MAE,Spread
2,CatBoost,14437.5,854.3
1,LightGBM,16164.8,982.3
0,XGBoost,17635.5,629.6


## Boosting libraries with default settings

Same cleaned training data, same preprocessing and same 5-fold split as before. None of these three is tuned.

| Model | MAE | Spread across folds |
|---|---|---|
| CatBoost | $14,438 | 854 |
| LightGBM | $16,165 | 982 |
| XGBoost | $17,636 | 630 |

For comparison: tuned Gradient Boosting $15,227, tuned Random Forest $16,583, Linear Regression baseline $17,207.

1. **CatBoost is ahead without any tuning.** It beats tuned Gradient Boosting by about $790, roughly one spread, so it is probably better but not certain.
2. **XGBoost is worse than the baseline.** Its default trees are deep and learn fast, which probably overfits on 1,166 houses. This explanation has not been tested.
3. **The comparison is not equal:** CatBoost, LightGBM and XGBoost are untuned, while Gradient Boosting and Random Forest were tuned. Next step: tune the three boosting libraries.

In [15]:
pipe_cb = Pipeline([('prep', make_preprocessor(X_train)), 
                    ('model', CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False))])

param_grid_cb = {
    "model__iterations" : [500, 1000],
    'model__learning_rate': [0.03, 0.1],
    'model__depth': [4, 6, 8]
}

search_cb = GridSearchCV(pipe_cb, param_grid_cb, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
search_cb.fit(X_train, y_train)

print(search_cb.best_params_)
print(-search_cb.best_score_)

{'model__depth': 6, 'model__iterations': 1000, 'model__learning_rate': 0.03}
14456.446145659846


In [24]:
cv_cb = pd.DataFrame(search_cb.cv_results_)
cv_cb['MAE'] = -cv_cb['mean_test_score']
view = cv_cb.sort_values('MAE')[['param_model__iterations', 'param_model__learning_rate','param_model__depth', 'MAE', 'std_test_score']]
print(view.round({'MAE': 1, 'std_test_score':1}).to_string())

    param_model__iterations  param_model__learning_rate  param_model__depth      MAE  std_test_score
6                      1000                        0.03                   6  14456.4           821.7
2                      1000                        0.03                   4  14478.3           831.6
5                       500                        0.10                   6  14652.6           771.2
7                      1000                        0.10                   6  14676.9           815.5
10                     1000                        0.03                   8  14699.4          1041.2
3                      1000                        0.10                   4  14779.7           604.2
4                       500                        0.03                   6  14796.6           907.9
1                       500                        0.10                   4  14896.8           621.6
0                       500                        0.03                   4  14977.4       

In [23]:
default_model_cb = Pipeline([('prep', make_preprocessor(X_train)),
                            ('model', CatBoostRegressor(random_state=42, verbose=0, allow_writing_files=False))])


default_model_cb.fit(X_train, y_train)
print(default_model_cb.named_steps['model'].get_all_params()['learning_rate'])

0.04194900020956993


## CatBoost: tuning result

Grid search over iterations (500, 1,000), learning rate (0.03, 0.1) and depth (4, 6, 8), with 5-fold cross-validation on the cleaned training set, scored by MAE.

| Setting | MAE |
|---|---|
| Untuned defaults (learning rate chosen automatically: 0.0419) | $14,438 |
| Best of 12 grid combinations (1,000 iterations, learning rate 0.03, depth 6) | $14,456 |

All 12 combinations fall within $549 of each other, which is less than the fold-to-fold spread ($600 to $1,180), so none is clearly better than another. The untuned model scored slightly better than every grid combination, because its automatic learning rate (0.0419) lies between the two values tested and the search started near a good setting.

**Final choice: CatBoost with default settings (MAE $14,438).** It is the best model so far, about $790 ahead of tuned Gradient Boosting ($15,227), a gap roughly equal to one spread, so it is probably better but not certain.

In [26]:
pipe_xgb = Pipeline([('prep', make_preprocessor(X_train)), 
                     ('model', XGBRegressor(random_state=42, n_jobs=-1))])

param_grid_xgb = {
    'model__n_estimators': [300, 600],
    'model__learning_rate': [0.03, 0.1],
    'model__max_depth': [2, 3, 4]
}

search_xgb = GridSearchCV(pipe_xgb, param_grid_xgb, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
search_xgb.fit(X_train, y_train)

print(search_xgb.best_params_)
print(-search_xgb.best_score_)

{'model__learning_rate': 0.1, 'model__max_depth': 2, 'model__n_estimators': 600}
14915.8734375


In [27]:
param_grid_xgb2 = {
    'model__n_estimators': [600, 1000],
    'model__learning_rate': [0.1, 0.2],
    'model__max_depth': [1, 2],
}

search_xgb2 = GridSearchCV(pipe_xgb,param_grid_xgb2 , cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
search_xgb2.fit(X_train, y_train)

print(search_xgb2.best_params_)
print(-search_xgb2.best_score_)

cv_xgb = pd.DataFrame(search_xgb2.cv_results_)
cv_xgb['MAE'] = -cv_xgb['mean_test_score']
view = cv_xgb.sort_values('MAE')[['param_model__n_estimators', 'param_model__learning_rate', 'param_model__max_depth', 'MAE', 'std_test_score']]
print(view.round({'MAE': 1, 'std_test_score': 1}).to_string())

{'model__learning_rate': 0.1, 'model__max_depth': 2, 'model__n_estimators': 1000}
14780.6681640625
   param_model__n_estimators  param_model__learning_rate  param_model__max_depth      MAE  std_test_score
3                       1000                         0.1                       2  14780.7           327.6
2                        600                         0.1                       2  14915.9           434.2
6                        600                         0.2                       2  15008.2           367.0
7                       1000                         0.2                       2  15125.5           272.2
1                       1000                         0.1                       1  15471.9           264.4
5                       1000                         0.2                       1  15521.2           486.8
4                        600                         0.2                       1  15652.3           332.4
0                        600                         


Model	MAE	Spread
2	CatBoost	14437.5	854.3
1	LightGBM	16164.8	982.3
0	XGBoost	17635.5	629.6


{'model__depth': 6, 'model__iterations': 1000, 'model__learning_rate': 0.03}
14456.446145659846
    param_model__iterations  param_model__learning_rate  param_model__depth      MAE  std_test_score
6                      1000                        0.03                   6  14456.4           821.7
2                      1000                        0.03                   4  14478.3           831.6
5                       500                        0.10                   6  14652.6           771.2
7                      1000                        0.10                   6  14676.9           815.5
10                     1000                        0.03                   8  14699.4          1041.2
3                      1000                        0.10                   4  14779.7           604.2
4                       500                        0.03                   6  14796.6           907.9
1                       500                        0.10                   4  14896.8           621.6
0                       500                        0.03                   4  14977.4          1073.1
8                       500                        0.03                   8  14986.4          1176.5
11                     1000                        0.10                   8  14993.6          1131.5
9                       500                        0.10                   8  15005.2          1147.6
0.04194900020956993



{'model__learning_rate': 0.1, 'model__max_depth': 2, 'model__n_estimators': 600}
14915.8734375
{'model__learning_rate': 0.1, 'model__max_depth': 2, 'model__n_estimators': 1000}
14780.6681640625
   param_model__n_estimators  param_model__learning_rate  param_model__max_depth      MAE  std_test_score
3                       1000                         0.1                       2  14780.7           327.6
2                        600                         0.1                       2  14915.9           434.2
6                        600                         0.2                       2  15008.2           367.0
7                       1000                         0.2                       2  15125.5           272.2
1                       1000                         0.1                       1  15471.9           264.4
5                       1000                         0.2                       1  15521.2           486.8
4                        600                         0.2                       1  15652.3           332.4
0                        600                         0.1                       1  15685.0           343.3

## XGBoost: tuning result

Two grid searches with 5-fold cross-validation on the cleaned training set, scored by MAE.

| Setting | MAE |
|---|---|
| Untuned defaults | $17,636 |
| Search 1 best (600 trees, learning rate 0.1, depth 2) | $14,916 |
| Search 2 best (1,000 trees, learning rate 0.1, depth 2) | $14,781 |

1. **Depth matters most.** Shallow trees were the fix: depth 2 beat depths 1, 3 and 4. Depth 1 was worse than depth 2 at every matching setting, by $400 to $770.
2. **Learning rate 0.1 beat 0.2** at every matching setting. More trees helped only at the lower rate (600 to 1,000 trees gained $135, less than one spread) and hurt at 0.2.
3. **The top three settings are effectively tied** (within $227, with a spread of about $330), so I stopped searching.

**Final choice: 1,000 trees, learning rate 0.1, depth 2 (MAE $14,781).** Tuning improved the MAE by about 16%, from worse than the linear baseline to second place behind default CatBoost ($14,438). The gap to CatBoost is about one XGBoost spread, so the two are effectively tied.